# HY-489 · Lab 2 — FHIR in practice

**Friday 16 October 2026 · 2 hours · GRADED, 8% of your final mark**

Deadline: **Thursday 22 October, 23:59.**

Names: _(both of you, if you paired)_

Student number used in the identifier namespace: _____

Before you submit: **restart the kernel and run all**, then run `python check_submission.py`.


---
## 0 · Setup  `0:00 – 0:10`


In [ ]:
import json, uuid
import pandas as pd, requests
from pathlib import Path

import sys; sys.path.insert(0, '..')
from src import fhirclient as fc
from src import mapping as mp

BASE = "https://hapi.fhir.org/baseR4"     # or "http://localhost:8080/fhir" for a local HAPI

cs = fc.get_json(f"{BASE}/metadata")
print(cs["fhirVersion"], cs.get("software", {}).get("name"))
print("namespace:", mp.SYSTEM)
assert not mp.SYSTEM.endswith("AM0000"), "Put your student number in src/mapping.py SYSTEM"


In [ ]:
DATA = "../lab01/data/"

patients   = pd.read_csv(DATA + "patients.csv",     parse_dates=["BIRTHDATE"])
encounters = pd.read_csv(DATA + "encounters.csv",   parse_dates=["START", "STOP"])
conditions = pd.read_csv(DATA + "conditions.csv",   parse_dates=["START"])
obs        = pd.read_csv(DATA + "observations.csv", parse_dates=["DATE"])

for name, df in [("patients", patients), ("encounters", encounters),
                 ("conditions", conditions), ("observations", obs)]:
    print(f"{name:<13}{df.shape}")


---
## Part A · Read  `0:10 – 0:40`

Before you query a server, read what it supports.

`fc.fetch_all` follows `link rel=next` and drops `search.mode == "include"` entries,
so what it returns is the matches and nothing else.


In [ ]:
rest = cs['rest'][0]
for r in rest['resource']:
    if r['type'] == 'Observation':
        print('interactions:', [i['code'] for i in r.get('interaction', [])])
        print('search params:', sorted(sp['name'] for sp in r.get('searchParam', []))[:20])


### ✅ Checkpoint 1 — read a server you did not build  `12 min`

1. Print the server's FHIR version and the search parameters it supports for `Observation`.
2. Fetch any one `Patient` by id. Print id, gender, birthDate and the number of identifiers.
3. Search for Patients with `gender=female` born on or after 1990-01-01. **How many matched, and how many did you receive?**
4. Pick a patient with data. Fetch their Observations with `_include=Observation:patient`, and count only the matches.
5. Find the same patient's Conditions. How many carry a `coding`, and how many are `text` only?

_Task 3 is the trap: `total` and `len(entry)` are different numbers._


In [ ]:
# 1.


In [ ]:
# 2.


In [ ]:
# 3.  report BOTH numbers and say which is which


In [ ]:
# 4.


In [ ]:
# 5.


**Checkpoint 1, in words.** How many matched vs how many you received, and your coding-vs-text ratio:

_your answer:_


---
## Part B · Map  `0:40 – 1:15`

Fill in the four functions in `src/mapping.py`, then run them from here.
Do not redefine them in the notebook: we import and test the module.

Reload after every edit:


In [ ]:
import importlib
importlib.reload(mp)

row = patients.iloc[0]
pt = mp.to_patient(row)
print(pt.json(indent=2))


### Work on 50 patients, not 1,174

You are proving the mapping works, not loading a warehouse.


In [ ]:
cohort = patients.head(50)
ids = set(cohort['Id'])

enc_c = encounters[encounters['PATIENT'].isin(ids)]
con_c = conditions[conditions['PATIENT'].isin(ids)]
obs_c = obs[obs['PATIENT'].isin(ids)]

print(len(cohort), len(enc_c), len(con_c), len(obs_c))


### Build, and keep every failure

A rejection table is a deliverable. Do not use `except: pass`.


In [ ]:
from pydantic import ValidationError

built, rejected = [], []

def build(fn, row, *args, kind=''):
    try:
        built.append((kind, fn(row, *args)))
    except (ValidationError, Exception) as e:
        msg = e.errors()[0]['msg'] if isinstance(e, ValidationError) else f'{type(e).__name__}: {e}'
        rejected.append({'kind': kind, 'source_id': row.get('Id', row.name), 'error': msg})

for _, r in cohort.iterrows():
    build(mp.to_patient, r, kind='Patient')

print(len(built), 'built,', len(rejected), 'rejected')


### ✅ Checkpoint 2 — map your Lab 1 data  `15 min`

1. Write `to_patient`, `to_encounter`, `to_condition` and `to_observation` in `src/mapping.py`.
2. Run them over the 50-patient cohort and everything belonging to those patients.
3. Collect every validation failure with its source key and the error message.
4. Report how many resources of each type you built, and how many were rejected and why.
5. For observations, report how many became `valueQuantity`, `valueCodeableConcept` and `valueString`.


In [ ]:
# 2. build encounters, conditions and observations for the cohort


In [ ]:
# 4. counts by type, built vs rejected


In [ ]:
# 5. the value[x] breakdown


In [ ]:
# write the rejection table - this file is marked
rej = pd.DataFrame(rejected, columns=['kind', 'source_id', 'error'])
rej.to_csv('../rejections.csv', index=False)
print(len(rej), 'rows written to rejections.csv')
rej.head(10)


---
## Part C · Write  `1:15 – 1:45`

Validate on the server first. `$validate` stores nothing.


In [ ]:
issues = fc.validate_resource(BASE, pt)
print(fc.issues_summary(issues))


### Build a transaction Bundle

Give each resource a `urn:uuid:` fullUrl, reference it from its children, and put
`ifNoneExist` on every Patient so the load can be re-run safely.


In [ ]:
def urn():
    return f'urn:uuid:{uuid.uuid4()}'

entries = []
for _, r in cohort.head(5).iterrows():
    pu = urn()
    ine = f"identifier={mp.SYSTEM}|{r['Id']}"
    entries.append((pu, json.loads(mp.to_patient(r).json()), 'Patient', ine))
    # TODO: encounters, conditions and observations for this patient,
    #       referencing pu as the subject

bundle = fc.transaction_bundle(entries)
print(len(bundle['entry']), 'entries')


### ✅ Checkpoint 3 — write it, and prove it  `12 min`

1. Run `$validate` on one Patient, one Condition and one Observation. Record every issue.
2. Build a transaction Bundle for five patients and all of their encounters, conditions and observations.
3. Use `urn:uuid:` internal references and `ifNoneExist` on every Patient entry.
4. POST it. Report the status of every entry, and save the source-key to server-id mapping.
5. Search by your identifier namespace and confirm the counts match what you sent.

_Post the Bundle twice. With `ifNoneExist` the second run creates no duplicate patients. Report what you observed._


In [ ]:
resp = fc.post_bundle(BASE, bundle)
for e in resp.get('entry', []):
    print(e['response']['status'], e['response'].get('location'))


In [ ]:
# 4. save the mapping from source key to server id - Lab 3 starts from this
# id_map = pd.DataFrame(...)
# id_map.to_csv('../id_map.csv', index=False)


In [ ]:
# 5. the independent check
mine = fc.get_json(f'{BASE}/Patient', params={'identifier': f'{mp.SYSTEM}|', '_count': 100})
print(mine.get('total'), 'patients under my namespace')


In [ ]:
# post the bundle a second time and compare


**What happened on the second POST?**

_your answer:_


---
## Wrap  `1:45 – 2:00`

### Submit

* this notebook, all three checkpoints answered, **restarted and run top to bottom**
* `src/mapping.py` with your four functions
* `rejections.csv`
* `NOTES.md`, ten to twenty lines

Then run:

```bash
python check_submission.py
```

**Deadline: Thursday 22 October, 23:59.**

### Marking

| Criterion | Marks |
|---|---|
| Reading (Checkpoint 1) | 20 |
| Mapping (Checkpoint 2) | 30 |
| Error handling | 15 |
| Writing (Checkpoint 3) | 20 |
| Notes and honesty | 15 |

### Before Lab 3 (Friday 30 October)

Keep `id_map.csv` and your count of text-only Conditions. Lab 3 starts from them.
